In [1]:
# !pip install --q --upgrade trl
!pip install --q --upgrade accelerate
!pip install --q transformers==4.57.1 
!pip install --q trl==0.24.0 
!pip install --q peft==0.17.1 
!pip install --q accelerate==1.11.0 
!pip install --q bitsandbytes==0.48.2


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
from datasets import load_dataset, Dataset
from datasets import concatenate_datasets
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
import bitsandbytes as bnb
import re
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from sklearn.model_selection import train_test_split

[2025-11-19 08:12:26,323] INFO numexpr.utils: Note: detected 128 virtual cores but NumExpr set to maximum of 64, check "NUMEXPR_MAX_THREADS" environment variable.
[2025-11-19 08:12:26,323] INFO numexpr.utils: Note: NumExpr detected 128 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 8.
[2025-11-19 08:12:26,323] INFO numexpr.utils: NumExpr defaulting to 8 threads.


In [3]:
model_name = "Qwen/Qwen2.5-7B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model.eval()

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=1e-06)
    (ro

In [4]:
import torch
print("Allocated:", torch.cuda.memory_allocated()/1024**3, "GB")
print("Reserved:", torch.cuda.memory_reserved()/1024**3, "GB")
print(next(model.model.layers[0].parameters()).dtype)

Allocated: 28.45729684829712 GB
Reserved: 56.322265625 GB
torch.float32


In [5]:
# ds_1 = load_dataset("chillies/IELTS-writing-task-2-evaluation")
ds_2 = load_dataset("chillies/IELTS_evaluations")
# ds_train = concatenate_datasets({ds_1["train"], ds_2["train"]})
# ds_test = concatenate_datasets({ds_1["test"], ds_2["test"]})
ds_train = ds_2["train"]
ds_test = ds_2["test"]
print(ds_train)
print(ds_test)

train.csv:   0%|          | 0.00/48.2M [00:00<?, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/9912 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/495 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'essay', 'evaluation', 'band'],
    num_rows: 9912
})
Dataset({
    features: ['prompt', 'essay', 'evaluation', 'band'],
    num_rows: 495
})


In [6]:
def normalizedData(data: dict):
    pattern = r"Suggested Overall Band Score:\s*(\d+(?:\.\d+)?)"
    scores = re.findall(pattern, data["evaluation"])
    
    if not scores:
        return data

    try:
        band_value = float(data["band"])
    except ValueError:
        return data

    diff = abs(float(scores[0]) - band_value)

    if diff == 1:
        data["band"] = str(band_value + 0.5)
    elif diff > 1:
        data["band"] = str(-1)
    else:
        data["band"] = str(band_value)

    return data
def is_valid_band(x):
    try:
        val = float(x["band"])
        return val != -1
    except ValueError:
        return True
ds_train = ds_train.map(normalizedData)
ds_train = ds_train.filter(is_valid_band)


Map:   0%|          | 0/9912 [00:00<?, ? examples/s]

Filter:   0%|          | 0/9912 [00:00<?, ? examples/s]

In [7]:
prompt_template_with_input = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

## Instruction:
{instruction}
## Input:
{input}
## Response:
{output}
"""

instruction_prompt ="""You are an IELTS Writing Task 2 examiner.
 Evaluate the essay below in detail according to the four official IELTS Writing criteria: Task Achievement, Coherence and Cohesion, Lexical Resource, and Grammatical Range and Accuracy.
 For each criterion, provide specific comments, examples, and a suggested band score.
 Then, provide an Overall Band Score and general feedback including Strengths, Areas for Improvement, and Suggestions for Enhancement.
 """


process_data = []
for data in ds_train :
  formatted_input = prompt_template_with_input.format(instruction = instruction_prompt,
                                                      input = f"Essay Topic:\n {data['prompt']}\n\nEssay:\n{data['essay']} ",
                                                      output =  f"Evaluation:\n{data['evaluation']}\n\nOverall Band Score :\n{data['band']}"
                                                      )
  process_data.append({"text": formatted_input})


In [8]:
process_data[1]

{'text': 'Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n## Instruction:\nYou are an IELTS Writing Task 2 examiner.\n Evaluate the essay below in detail according to the four official IELTS Writing criteria: Task Achievement, Coherence and Cohesion, Lexical Resource, and Grammatical Range and Accuracy.\n For each criterion, provide specific comments, examples, and a suggested band score.\n Then, provide an Overall Band Score and general feedback including Strengths, Areas for Improvement, and Suggestions for Enhancement.\n \n## Input:\nEssay Topic:\n Young people who commit crimes should be treated in the same way as adults who commit crimes. To what extend do you agree or disagree?\n\nEssay:\nIn this modern era, youngsters who commit offences should be punished in  similar ways as mature people. I do think that younger criminals should have behaved less strictly than older

In [9]:
def clean_text(text: str) -> str:
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ ]{2,}', ' ', text)
    text = '\n'.join(line.strip() for line in text.splitlines())
    text = re.sub(r'[ \t]+', ' ', text)  
    return text.strip()


for i in range (len(process_data)) :
    process_data[i]["text"] = clean_text(process_data[i]["text"])

In [10]:
train_indices, val_indices = train_test_split(range(len(process_data)), test_size=0.1, random_state=42)
process_dataset = Dataset.from_list(process_data)
train_dataset = process_dataset.select(train_indices)
val_dataset = process_dataset.select(val_indices)

In [11]:
lora_config = LoraConfig (
  r =8 , # Rank
  lora_alpha =32 , # Alpha parameter
  target_modules =[ # Target attention matrices
  "q_proj",
  "v_proj",
  ],
  lora_dropout =  0.05,
  bias = "none",
  task_type = "CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,523,136 || all params: 7,618,139,648 || trainable%: 0.0331


In [12]:
!pip install --q wandb



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [30]:
import wandb
import os
wandb.login(key='ec2a5e1f6687dc8ae6c84e4f664590cc787408a1')
os.environ["WANDB_DISABLED"] = "true"


wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


In [15]:
from trl import SFTConfig
sft_config = SFTConfig(
      output_dir = "output",
      num_train_epochs = 3,
      per_device_train_batch_size = 4,
      gradient_accumulation_steps= 4,
      learning_rate = 4e-6,
      bf16=True,
      save_total_limit = 3,
      logging_steps = 100,
      eval_steps = 100,
      optim="paged_adamw_8bit",
      packing=False,
      warmup_ratio = 0.1,  
      completion_only_loss=True,
)
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=sft_config,
)

trainer.train()


Adding EOS to train dataset:   0%|          | 0/7119 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/7119 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/7119 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/792 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/792 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/792 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


UsageError: api_key not configured (no-tty). call wandb.login(key=[your_api_key])

In [ ]:
ls -R output/


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model = "Qwen/Qwen2.5-7B-Instruct"
adapter = "Thaiphu/Qwen2.5_Ielts_task_2"

tokenizer = AutoTokenizer.from_pretrained(base_model)
model = AutoModelForCausalLM.from_pretrained(base_model, device_map="auto")
model = PeftModel.from_pretrained(model, adapter)

prompt = """Evaluate the following IELTS Task 2 essay in detail based on the 4 criteria:
Topic: Some people think universities should provide
